# 🔬 Notebook 3: Deep Dive — Priority, Retries, Idempotency

For each topic we show the same **bad → better → best** pattern so you can *feel* why each improvement exists. Every cell runs in pure Python — no external services.

## 🛠️ Setup

```bash
cd 06-system-designs/notification-system
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Priority queues

Scenario: the marketing team pushes **1,000,000** promo messages. A second later, a user
requests a 2FA code. How do we make sure the 2FA code doesn't wait in line behind a million
promos?

### ❌ Bad — a single FIFO queue

Everything shares one line. High-priority messages are stuck behind low-priority ones.

In [ ]:
from collections import deque

q = deque()
for i in range(1_000_000): q.append(("low",  f"promo-{i}"))
q.append(("high", "2FA-code-for-alice"))

# Pop the first 3 — the 2FA code is a million items away.
print("first 3 popped:", [q.popleft() for _ in range(3)])
print("index of 2FA code in queue:", 1_000_000 - 3)


### ⚠️ Better — strict priority (always drain high first)

Multiple queues, and the worker *always* pulls `high` before `normal` before `low`.
Problem: if `high` traffic is continuous, **low/normal starve forever**.

In [ ]:
from collections import deque

# Starvation only shows up when high-priority work keeps ARRIVING. A finite burst of
# 2FA codes drains and everyone else gets served — which is why the naive demo of this
# looks fine. Model a steady 2FA stream instead: 3 new high-priority messages per tick.
HIGH_ARRIVALS_PER_TICK = 7      # a steady 2FA stream that fully consumes our capacity
TICKS = 12

def fresh_queues():
    qs = {"high": deque(), "normal": deque(), "low": deque()}
    for i in range(20): qs["low"].append(f"promo-{i}")
    for i in range(20): qs["normal"].append(f"receipt-{i}")
    return qs

def arrive(qs, tick):
    for i in range(HIGH_ARRIVALS_PER_TICK):
        qs["high"].append(f"2fa-{tick}-{i}")

def strict_pop(qs):
    for level in ("high", "normal", "low"):
        if qs[level]:
            return level, qs[level].popleft()
    return None

qs = fresh_queues()
served = {"high": 0, "normal": 0, "low": 0}
CAPACITY = 7                      # the worker pool can do 7 sends per tick
for tick in range(TICKS):
    arrive(qs, tick)
    for _ in range(CAPACITY):
        got = strict_pop(qs)
        if got: served[got[0]] += 1

print(f"strict priority after {TICKS} ticks: {served}")
print(f"  backlog left: high={len(qs['high'])} normal={len(qs['normal'])} low={len(qs['low'])}")
print("⚠️  normal and low were served ZERO times. Not 'delayed' — starved. As long as high")
print("    arrives at or above our capacity, they will never be served at all.")


### ✅ Best — weighted round-robin (starvation-free)

Pull a fixed **ratio** per round: e.g. 4 high : 2 normal : 1 low.
Low priority still makes progress, but always loses the fight for throughput.

In [ ]:
class WeightedScheduler:
    """Serve a fixed RATIO per round instead of draining strictly by level."""
    def __init__(self, ratio):
        self.queues = {k: deque() for k in ratio}
        self.ratio  = ratio
    def push(self, level, msg):
        self.queues[level].append(msg)
    def drain_round(self, budget):
        """Take up to `ratio[level]` from each level, capped by total `budget`."""
        out = []
        for level, n in self.ratio.items():
            for _ in range(n):
                if len(out) >= budget:
                    return out
                if self.queues[level]:
                    out.append((level, self.queues[level].popleft()))
        # NOTE: if a level has no work, this toy wastes its slots. Real implementations
        # (deficit round robin) backfill the unused budget from whoever still has a queue.
        return out

# EXACT same workload and capacity as the strict-priority run above.
s = WeightedScheduler({"high": 4, "normal": 2, "low": 1})   # 4:2:1, sums to CAPACITY
for i in range(20): s.push("low", f"promo-{i}")
for i in range(20): s.push("normal", f"receipt-{i}")

served_w = {"high": 0, "normal": 0, "low": 0}
for tick in range(TICKS):
    for i in range(HIGH_ARRIVALS_PER_TICK):
        s.push("high", f"2fa-{tick}-{i}")
    for level, _msg in s.drain_round(CAPACITY):
        served_w[level] += 1

print(f"weighted round-robin after {TICKS} ticks: {served_w}")
print(f"  backlog left: high={len(s.queues['high'])} "
      f"normal={len(s.queues['normal'])} low={len(s.queues['low'])}")
assert all(v > 0 for v in served_w.values()), "WRR must serve every level"
print("✅ every level got served under the identical high-priority pressure that starved")
print("   normal and low completely under strict priority.")
print("""
The cost, stated plainly: high priority is now SLOWER than it would be under strict
priority, because we deliberately spend 1 of every 7 slots on a marketing promo and 2 more
on receipts while a 2FA code waits. The high backlog above is the price of the low backlog draining. If your
high-priority SLA cannot absorb that, the answer is not 'go back to strict priority' —
it is to add capacity, or to shed low-priority work outright rather than queue it.""")


## 2. Retries

Scenario: Twilio returns `503 Service Unavailable`. We want to retry — but *how* matters a lot.

### ❌ Bad — immediate retry loop

Hammers the already-struggling provider, makes the outage worse (**thundering herd**).

In [ ]:
import random
random.seed(4)   # a seed where the first few attempts fail, for a clearer demo

def flaky_send():                         # pretend provider: ~70% fail
    return "OK" if random.random() > 0.7 else "FAIL"

attempts = 0
while True:
    attempts += 1
    if flaky_send() == "OK":
        print(f"succeeded on attempt {attempts}"); break
    print(f"  attempt {attempts} failed → retry IMMEDIATELY (no delay)")
    if attempts >= 8:
        print("gave up"); break
print(f"❌ hit the provider {attempts} times back-to-back — thundering herd")


### ⚠️ Better — exponential backoff

Wait longer between attempts: `1s, 2s, 4s, 8s, …`. Gentler on the provider, but if
1,000 workers all failed at the same moment they'll *all* retry at t=1s, t=2s, … —
a synchronized stampede.

In [ ]:
import random, time
random.seed(3)   # seed that fails a few times first to show the backoff

def backoff_exp(attempt, base=0.05, cap=1.0):     # no jitter
    return min(cap, base * (2 ** attempt))

MAX = 8
start = time.time()
attempts = 0
while attempts < MAX:
    if flaky_send() == "OK":
        print(f"OK after {attempts+1} attempts, elapsed {time.time()-start:.2f}s"); break
    wait = backoff_exp(attempts)
    print(f"  attempt {attempts+1} failed → sleep {wait:.3f}s (doubles every time)")
    time.sleep(wait)
    attempts += 1


### ✅ Best — exponential backoff **with full jitter** + **DLQ**

Two fixes:

1. **Jitter**: sleep a *random* time in `[0, exp_wait]` so retries spread out.
2. **Dead-letter queue**: after N failed attempts, park the message for human review
   instead of retrying forever.

In [ ]:
import random, time
random.seed(1)

def backoff_jitter(attempt, base=0.01, cap=1.0):
    """Full jitter: sleep a RANDOM time in [0, exp_wait] so a fleet of retriers spreads out."""
    return random.uniform(0, min(cap, base * (2 ** attempt)))

MAX_ATTEMPTS = 4
dlq = []

def send_with_retry(msg):
    for attempt in range(MAX_ATTEMPTS):
        if flaky_send() == "OK":
            return f"{msg}: OK on attempt {attempt+1}"
        time.sleep(backoff_jitter(attempt))
    dlq.append(msg)                    # bounded retries: park it, don't loop forever
    return f"{msg}: → DLQ after {MAX_ATTEMPTS} attempts"

for m in [f"m{i}" for i in range(1, 9)]:
    print(send_with_retry(m))
print(f"\nDLQ ({len(dlq)}): {dlq}")
print("✅ bounded retries: the provider is protected, and nothing is silently lost —")
print("   DLQ contents are a queue depth you alert on and a human replays.")


### Seeing what jitter is actually for

The retry loop above doesn't show jitter's benefit, because one client retrying alone doesn't
care when it retries. Jitter exists for the **fleet**: 1,000 workers whose calls all failed at
the same instant (because the provider had one bad second) will, without jitter, all retry at
exactly t=1s, then all at t=2s — reproducing the original spike on a struggling provider.

In [ ]:
import random
from collections import Counter
random.seed(7)

FLEET = 1_000
def retry_time_no_jitter(attempt, base=1.0):
    return base * (2 ** attempt)
def retry_time_full_jitter(attempt, base=1.0):
    return random.uniform(0, base * (2 ** attempt))

for attempt in (0, 1):
    plain  = Counter(round(retry_time_no_jitter(attempt), 1)   for _ in range(FLEET))
    jitter = Counter(round(retry_time_full_jitter(attempt), 1) for _ in range(FLEET))
    print(f"attempt {attempt + 1}: all {FLEET} workers retry within a 0.1s window...")
    print(f"  no jitter  : peak {max(plain.values()):>5} workers in one 0.1s bucket "
          f"(all at t={list(plain)[0]}s)")
    print(f"  full jitter: peak {max(jitter.values()):>5} workers in one 0.1s bucket, "
          f"spread over {len(jitter)} buckets")
print("\nWithout jitter the retry storm is as sharp as the original traffic spike — you have")
print("built a system that reliably kicks a provider while it is down. With full jitter the")
print("same load arrives smeared across the whole backoff window.")


## 3. Idempotency

Scenario: caller's HTTP client times out after 30s, so it retries. Our service actually
received the first request and queued it. Without idempotency we send **two** receipts.

### ❌ Bad — no dedup, just send

User gets two "Your order shipped!" push notifications. 🤦

In [ ]:
sent = []
def naive_send(payload): sent.append(payload)

naive_send({"user":42,"msg":"order shipped"})
naive_send({"user":42,"msg":"order shipped"})   # caller retry
print("sent:", sent)
print(f"❌ user got {len(sent)} copies")


### ⚠️ Better — hash the payload

Hash the whole payload and skip duplicates. Works for accidental retries, but:

- Any tiny difference (e.g. timestamp field) → different hash → still duplicates.
- Intentional resends (customer clicked "resend receipt") get silently dropped.

In [ ]:
import hashlib, json
seen_hashes = set()
def hash_send(payload):
    h = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    if h in seen_hashes: return "duplicate"
    seen_hashes.add(h)
    return "sent"

p = {"user":42, "msg":"order shipped"}
print(hash_send(p))                                    # sent
print(hash_send(p))                                    # duplicate ✔
print(hash_send({**p, "ts": 1700000000}))              # ❌ timestamp makes it "new"


### ✅ Best — caller-provided `dedup_key` with TTL

The caller picks a **meaningful** key (`order-123-shipped`). We store it with a TTL long
enough to cover any reasonable retry window (e.g. 7 days). Same key → no-op. Different
key → genuinely different send, even if the payload looks similar.

In [ ]:
import time

class DedupStore:
    def __init__(self, ttl_seconds=7*24*3600):
        self.ttl = ttl_seconds
        self._d  = {}                                  # key -> (first_seen_ts, payload)

    def _evict(self):
        now = time.time()
        for k in [k for k,(ts,_) in self._d.items() if now - ts > self.ttl]:
            del self._d[k]

    def enqueue(self, dedup_key, payload):
        self._evict()
        if dedup_key in self._d:
            return "DUPLICATE — not re-sent"
        self._d[dedup_key] = (time.time(), payload)
        return "NEW — queued"

store = DedupStore()
print(store.enqueue("order-123-shipped", {"m":"Your order shipped!"}))
print(store.enqueue("order-123-shipped", {"m":"Your order shipped!"}))   # retry
print(store.enqueue("order-123-resent",  {"m":"Your order shipped!"}))   # support resent
print(store.enqueue("order-124-shipped", {"m":"Different order!"}))
print("✅ retries dedupe, intentional resends get through")


## 4. At-least-once delivery — the duplicate the dedup_key does *not* catch

Everything in §3 dedupes at the **front door**: two `POST /notify` calls with the same
`dedup_key` collapse into one queued message. Good. But that is not where most duplicate
notifications come from.

Every durable queue worth using — SQS, Kafka, RabbitMQ, Redis Streams — delivers **at least
once**. A message is handed to a worker with a visibility timeout; if the worker does not
**ack** before the timeout, the queue assumes the worker died and hands the same message to
someone else. Now consider the worker's two steps:

```
   1. call the provider   (APNs / Twilio / SES)   <- side effect leaves our system
   2. ack the queue                               <- tells the queue we're done
```

If the process dies **between** step 1 and step 2 — an OOM kill, a deploy, a 31-second
network stall — the push has already gone out and the queue never heard about it. The message
comes back. The user gets a second notification.

**There is no ordering of those two steps that fixes it.** Ack first and a crash loses the
notification entirely (at-most-once). Ack second and a crash duplicates it (at-least-once).
You are choosing which failure you prefer, and for notifications "sent twice" beats "never
sent". Let's make the duplicate happen, then kill it.

In [ ]:
import time
from collections import deque

class Queue:
    """A toy at-least-once queue: visibility timeout + explicit ack, like SQS."""
    def __init__(self, visibility_timeout=0.2):
        self._ready = deque()
        self._inflight = {}                      # receipt -> (msg, deadline)
        self._next_receipt = 1
        self.vt = visibility_timeout
        self.deliveries = 0                      # how many times we handed out a message

    def send(self, msg):
        self._ready.append(msg)

    def receive(self):
        # Anything whose visibility timeout expired goes back on the ready queue.
        now = time.monotonic()
        for r in [r for r, (_, dl) in self._inflight.items() if dl < now]:
            msg, _ = self._inflight.pop(r)
            self._ready.append(msg)
        if not self._ready:
            return None, None
        msg = self._ready.popleft()
        receipt = self._next_receipt
        self._next_receipt += 1
        self._inflight[receipt] = (msg, now + self.vt)
        self.deliveries += 1
        return receipt, msg

    def ack(self, receipt):
        self._inflight.pop(receipt, None)

# The provider is the source of truth for "did the user actually get it".
pushes_sent = []
def provider_push(msg):
    pushes_sent.append(msg["dedup_key"])
    return "delivered"

def worker(q, crash_after_send_once=False, guard=None):
    """Drain the queue. `guard` (if given) is the delivery-side dedup store, and we CLAIM
    the key in it *before* calling the provider — the ordering matters, see below."""
    crashed = False
    while True:
        receipt, msg = q.receive()
        if msg is None:
            break
        if guard is not None:
            if msg["dedup_key"] in guard:         # someone already handed this to the provider
                q.ack(receipt)                    # so just retire the redelivery
                continue
            guard.add(msg["dedup_key"])           # claim it (Redis SETNX) BEFORE the side effect
        provider_push(msg)                        # <-- side effect leaves our system
        if crash_after_send_once and not crashed:
            crashed = True
            time.sleep(q.vt + 0.05)               # die before acking; visibility expires
            continue                              # (a new worker picks the message up)
        q.ack(receipt)

msg = {"user": 42, "dedup_key": "order-123-shipped", "body": "Your order shipped!"}

# --- Without a delivery-side guard: one enqueue, two pushes.
pushes_sent.clear()
q = Queue()
q.send(msg)
worker(q, crash_after_send_once=True)
print(f"no guard  : enqueued 1, queue delivered {q.deliveries}x, "
      f"user received {len(pushes_sent)} push(es)  {pushes_sent}")
print("            ^ the caller sent ONE request with ONE dedup_key and the user still got two.")


### The fix: dedupe again at the moment of delivery

The front-door dedup store answered *"have I accepted this request before?"*. We need a second
store answering a different question: *"have I already handed this to the provider?"* — checked
by the worker, immediately before the call.

In [ ]:
delivered_keys = set()          # in production: Redis SETNX, or a UNIQUE row in send_log

pushes_sent.clear()
q = Queue()
q.send(msg)
worker(q, crash_after_send_once=True, guard=delivered_keys)
print(f"with guard: enqueued 1, queue delivered {q.deliveries}x, "
      f"user received {len(pushes_sent)} push(es)  {pushes_sent}")

print("""
Read the honest fine print, because an interviewer will push on exactly this:

* The ORDER of the two lines in the worker is the whole design. We claim the key *before*
  calling the provider. Swap them — send first, record after — and re-run: the crash happens
  between the send and the record, the guard is still empty when the message comes back, and
  you get two pushes again. (Try it. That version is what most people write first.)

* Claim-before-send NARROWS the window, it does not close it. It converts "duplicate on crash"
  into "silent drop on crash": if we claim the key and then die before the provider actually
  accepts the request, nobody will ever retry it. You are always moving the failure, never
  deleting it. For notifications that bias is usually right — a duplicate annoys, a drop
  breaks 2FA — but it is a choice, so make it per channel and write it down.

* The only real exactly-once is PROVIDER-side idempotency. SendGrid, Twilio and Stripe all
  accept an idempotency key on the request; when they do, pass your dedup_key straight through
  and let the provider collapse the retry. That converts "at-least-once + our best effort"
  into effectively-once, and it is the answer the interviewer is fishing for.

* Anything without an idempotency key (plain SMTP, most APNs setups) cannot be made
  exactly-once at any price. Say so rather than hand-waving.

* Both dedup stores need a TTL, and the TTL must exceed the maximum retry window from §2
  (max attempts x max backoff). A TTL shorter than the retry window means a late retry sails
  straight past the guard.""")


## Recap

| Concern | ❌ Bad | ⚠️ Better | ✅ Best |
|---|---|---|---|
| Priority | single FIFO | strict priority | weighted round-robin |
| Retries | tight loop | exponential backoff | exp backoff + jitter + DLQ |
| Idempotency (front door) | none | hash payload | caller `dedup_key` + TTL |
| Duplicate delivery (at-least-once) | none — user gets 2 pushes | delivery-side guard before the provider call | provider-side idempotency key |

The last row is the one people forget: front-door idempotency and delivery-side
idempotency are **different stores answering different questions**, and you need both.

In **Notebook 4** we tackle the remaining production concerns:
**fan-out** (one event → many channels), **per-provider rate limiting**, and **circuit
breakers** to fail fast when a provider is down.